# MCP (Model Context Protocol)

MCP lets an agent use tools that live in a different program (or on a different computer). The agent
finds the tools when it connects. This notebook connects an agent to a small MCP server, the Hidden Leaf
Ninja Registry, two ways: over stdio (a local program) and over Streamable HTTP (the web). Then it shows
`HostedMCPTool`, where OpenAI connects for you. The MCP part needs no `OPENAI_API_KEY`. Only the live model turn does.


## Write a small MCP server

It has two tools: `get_ninja_rank`, and `check_mission_status` (a tool we hide later with a filter).
The server code is saved into a temp file, so the notebook stays self-contained.


In [1]:
import sys  # uses the same Python program that this notebook is running
import tempfile  # writes the demo MCP server into a temporary folder
from pathlib import Path  # builds a real file path to that temporary file

SERVER_SOURCE = """
from mcp.server.mcpserver import MCPServer

server = MCPServer(name="Hidden Leaf Ninja Registry")


# MCP tool: the one the demos really call
@server.tool()
def get_ninja_rank(ninja_name: str) -> str:
    "Look up a ninja's rank in the Village registry."
    return {"Naruto Uzumaki": "S", "Sasuke Uchiha": "S"}.get(ninja_name, "unknown")


# MCP tool: a restricted tool that the filter demo hides
@server.tool()
def check_mission_status(mission_id: str) -> str:
    "Check a mission's status (a restricted tool, filtered out later)."
    return f"{mission_id}: danger level 9, evacuate"


if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "http":
        server.run(transport="streamable-http", host="127.0.0.1", port=int(sys.argv[2]))
    else:
        server.run(transport="stdio")
"""

server_dir = Path(tempfile.mkdtemp(prefix="ninja_mcp_"))
server_path = server_dir / "ninja_mcp_server.py"
server_path.write_text(SERVER_SOURCE)

# params: how to start the server as a subprocess: the Python program and the server script
stdio_params = {"command": sys.executable, "args": [str(server_path)]}
print("[wrote server]", server_path)

[wrote server] /var/folders/3s/4kzjy6kd0h58f0cv6d05z7cr0000gn/T/ninja_mcp_tg66gyu4/ninja_mcp_server.py


## Connect with `MCPServerStdio` and use it in an `Agent`

The SDK starts the server as a small program and talks to it through its input and output (stdin and stdout).
`list_tools()` and `call_tool()` work on the server directly. Pass the connected server to
`Agent(mcp_servers=[...])`, and the agent calls its tools like any other tool.
`create_static_tool_filter(blocked_tool_names=[...])` hides a tool from the model completely.


In [2]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.mcp import (
    MCPServerStdio,  # connects to an MCP server running as a local program
    create_static_tool_filter,  # makes a fixed list of allowed or blocked tools for a server
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: connect over stdio, call a tool directly, let an agent call it, then hide a tool with a filter
async def demo_stdio() -> None:
    # Initialize the MCP server
    async with MCPServerStdio(
        params=stdio_params,
        name="Hidden Leaf Ninja Registry",
    ) as server:
        print("[tools over stdio]", [tool.name for tool in await server.list_tools()])
        call_result = await server.call_tool("get_ninja_rank", {"ninja_name": "Naruto Uzumaki"})
        # proves the subprocess connection works with no agent yet
        print("[call_tool]", call_result.content)

        # Initialize the agent
        agent = Agent(
            name="Leaf Village Assistant",
            instructions="Use the Hidden Leaf Ninja Registry to answer questions.",
            # mcp_servers: connected servers whose tools this agent can call
            mcp_servers=[server],
        )
        # Script the model
        model = ScriptedModel(
            [
                [function_call("get_ninja_rank", {"ninja_name": "Naruto Uzumaki"}, call_id="c1")],
                [assistant_message("Naruto Uzumaki is Kage-level, per the Village registry.")],
            ]
        )
        # Run the Agent
        result = await Runner.run(
            agent,
            "What rank is Naruto Uzumaki?",
            run_config=RunConfig(model=model, workflow_name="e13-mcp-agent"),
        )
        # proves an Agent calls an MCP tool through mcp_servers=[...] like any other tool
        print("[final_output]", result.final_output)


await demo_stdio()
# Outside a notebook: asyncio.run(demo_stdio())

[tools over stdio] ['get_ninja_rank', 'check_mission_status']
[call_tool] [TextContent(type='text', text='S', annotations=None, meta=None)]
[final_output] Naruto Uzumaki is Kage-level, per the Village registry.


In [3]:
async def demo_stdio_tool_filter() -> None:
    # Initialize the MCP server
    async with MCPServerStdio(
        params=stdio_params,
        name="Hidden Leaf Ninja Registry (filtered)",
        # tool_filter: hides check_mission_status completely instead of refusing it
        tool_filter=create_static_tool_filter(blocked_tool_names=["check_mission_status"]),
    ) as filtered_server:
        # proves the filter removes the tool from the list the model sees
        print("[filtered tool list]", [tool.name for tool in await filtered_server.list_tools()])


await demo_stdio_tool_filter()
# Outside a notebook: asyncio.run(demo_stdio_tool_filter())

[filtered tool list] ['get_ninja_rank']


## Connect over the network with `MCPServerStreamableHttp`

It is the same server, started as a web program. This is how you reach a real remote MCP server.
The only change is `params={"url": ...}` instead of a command.


In [4]:
import subprocess  # starts the streamable-HTTP server as a real separate program
import sys  # uses the same Python program that this notebook is running
import time  # a short wait while the HTTP server program starts

from agents.mcp import (
    MCPServerStreamableHttp,  # connects to an MCP server over the web (HTTP)
)


# Demo: run the same server as an HTTP program and call a tool over the network
async def demo_streamable_http() -> None:
    # Initialize the MCP server
    # Start the server in the background: Popen runs another program without waiting for it
    process = subprocess.Popen(
        # command: this notebook's Python, the server file, "http" mode, and port 8991
        [sys.executable, str(server_path), "http", "8991"],
        # DEVNULL: throws away the server's messages so they don't clutter the notebook
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # Wait for the server to start
    time.sleep(1.5)
    try:
        async with MCPServerStreamableHttp(
            # params: the web address to reach, instead of a command and args
            params={"url": "http://127.0.0.1:8991/mcp"},
            name="Hidden Leaf Ninja Registry (HTTP)",
        ) as server:
            print("[tools over streamable HTTP]", [tool.name for tool in await server.list_tools()])
            call_result = await server.call_tool("get_ninja_rank", {"ninja_name": "Sasuke Uchiha"})
            # proves the same server and tools work over a real HTTP connection, not only stdio
            print("[call_tool]", call_result.content)
    finally:
        # Stop the server: terminate() asks it to quit, wait() pauses until it has quit
        process.terminate()
        process.wait(timeout=5)


await demo_streamable_http()
# Outside a notebook: asyncio.run(demo_streamable_http())

[tools over streamable HTTP] ['get_ninja_rank', 'check_mission_status']
[call_tool] [TextContent(type='text', text='S', annotations=None, meta=None)]


## `HostedMCPTool` — OpenAI-hosted MCP connections

Here OpenAI's own computers connect to the MCP server for you, instead of your program doing it.
It goes in `tools=[...]`, not `mcp_servers=[...]`. Using it needs a real key and an MCP server address that
anyone can reach (a `localhost` server will not work). First we only build the tool, then the next cells run it live.


In [5]:
from agents import (
    HostedMCPTool,  # asks OpenAI's servers to connect to an MCP server for you
)

hosted_mcp_tool = HostedMCPTool(
    # tool_config: the settings OpenAI needs to reach the MCP server
    tool_config={
        "type": "mcp",
        # server_label: your own name for this connection
        "server_label": "hidden-leaf-ninja-registry",
        # server_url: a real MCP server that can be reached, in production
        "server_url": "https://example.com/mcp",
        # require_approval: "never" skips asking a human before each tool call
        "require_approval": "never",
    }
)
print("[constructed]", type(hosted_mcp_tool).__name__)

[constructed] HostedMCPTool


## Give `HostedMCPTool` to an `Agent` and run it — optional, needs a real key

No connection code and no `async with` here. We hand the tool to `Agent(tools=[...])`, and OpenAI lists and
calls the server's tools by itself, inside one model turn. This uses a real public MCP server (`gitmcp.io`).
`"require_approval": "never"` lets the model call tools without asking a human first. Skipped when there is no `OPENAI_API_KEY`.


In [6]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    HostedMCPTool,  # asks OpenAI's servers to connect to an MCP server for you
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real run where OpenAI itself connects to a public MCP server and calls its tools
async def demo_hosted_mcp() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="Use the MCP tools to answer questions about the repo. Answer briefly.",
        tools=[
            HostedMCPTool(
                # tool_config: the settings OpenAI needs to reach the MCP server
                tool_config={
                    "type": "mcp",
                    # server_label: your own name for this connection
                    "server_label": "gitmcp",
                    # server_url: a public MCP server that OpenAI can reach
                    "server_url": "https://gitmcp.io/openai/codex",
                    # require_approval: "never" skips asking a human before each tool call
                    "require_approval": "never",
                }
            )
        ],
    )
    # Run the Agent
    result = await Runner.run(agent, "What is this repo about? Answer in two sentences.")
    # Inspect the result
    print("[item types]", [item.type for item in result.new_items])
    print("[final_output]", result.final_output)


await demo_hosted_mcp()
# Outside a notebook: asyncio.run(demo_hosted_mcp())

[item types] ['mcp_list_tools_item', 'tool_call_item', 'message_output_item']
[final_output] This repo is OpenAI Codex, a Rust-based command-line coding agent. It provides interactive and non-interactive workflows for understanding, modifying, and running code with sandboxing, approvals, configuration, and execution policies.


## Run it live — optional, needs a real key

A real model picks the tool calls over stdio. With `check_mission_status` blocked, the model cannot call a
tool it never saw. Which tools the model calls is up to the model, so the cell prints the calls it made.
Skipped when there is no `OPENAI_API_KEY`.


In [9]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.mcp import (
    MCPServerStdio,  # connects to an MCP server running as a local program
    create_static_tool_filter,  # makes a fixed list of allowed or blocked tools for a server
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real run over stdio, then a real run where the filter hides check_mission_status
async def demo_live_stdio() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the MCP server
    async with MCPServerStdio(
        params=stdio_params,
        name="Registry (filtered)",
        # tool_filter: hides check_mission_status so the model never sees it
        tool_filter=create_static_tool_filter(blocked_tool_names=["check_mission_status"]),
    ) as server:
        # Initialize the agent
        agent = Agent(
            name="Leaf Village Assistant",
            instructions="Use the Hidden Leaf Ninja Registry tools to answer questions. Answer briefly.",
            mcp_servers=[server],
        )
        for question in [
            "What rank is Naruto Uzumaki?",
            "What's the status of Mission-07? use the check_mission_status tool",
        ]:
            # Run the Agent
            result = await Runner.run(agent, question)
            # Inspect the result
            called = [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"]
            # proves a real model finds MCP tools by itself, and cannot call the one the filter hid
            print(f"[live] {question} | tools called: {called} | {result.final_output}")


await demo_live_stdio()
# Outside a notebook: asyncio.run(demo_live_stdio())

[live] What rank is Naruto Uzumaki? | tools called: ['get_ninja_rank'] | Naruto Uzumaki’s rank is **S**.
[live] What's the status of Mission-07? use the check_mission_status tool | tools called: [] | I don’t have access to a `check_mission_status` tool—only the ninja rank registry is available.


In [10]:
import shutil  # deletes the temporary server file or folder

shutil.rmtree(server_dir, ignore_errors=True)
print("[cleaned up]", server_dir)

[cleaned up] /var/folders/3s/4kzjy6kd0h58f0cv6d05z7cr0000gn/T/ninja_mcp_tg66gyu4


## Summary

- `MCPServerStdio` starts a server program and `MCPServerStreamableHttp` connects over
  the web. `mcp_servers=[...]` on an `Agent` lets it call their tools like any other tool,
  and `create_static_tool_filter` hides tools from the model.
- `HostedMCPTool` goes in `tools=[...]` and lets OpenAI's servers connect to a public MCP server for you,
  with no connection code on your side (a live key is needed).
